# Movie Rating Prediction - Feature Engineering

This notebook transforms raw movie data into machine learning-ready features based on insights from our exploratory data analysis.

## Overview

Based on our EDA findings, we'll create features for:
- **Genre encoding** (binary and multi-hot)
- **Temporal features** (movie age, decade, seasonal effects)
- **Popularity metrics** (rating count bins, user engagement)
- **Rating pattern features** (variance, distribution characteristics)
- **Content features** (title length, genre combinations)
- **Collaborative features** (user-movie interactions)

## Target Variable
We'll predict **average movie ratings** from the MovieLens dataset (0.5-5.0 scale).

## 1. Import Libraries and Load Data

In [1]:
# Import essential libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Machine learning libraries
from sklearn.preprocessing import StandardScaler, LabelEncoder, MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Set up plotting
plt.style.use('default')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("🔧 FEATURE ENGINEERING SETUP")
print("=" * 40)
print("✅ Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

🔧 FEATURE ENGINEERING SETUP
✅ Libraries imported successfully!
Pandas version: 2.3.3
NumPy version: 2.3.3


In [2]:
# Load the IMDb data (for pre-release features)
import os

data_dir = '../data/processed'
raw_dir = '../data/raw'

print("📊 LOADING IMDb DATA")
print("=" * 30)

# Load IMDb processed data
try:
    # Load the combined dataset that has all basic info + ratings
    movies_df = pd.read_parquet(f'{data_dir}/imdb_combined_basic.parquet')
    
    # Load crew information (directors, writers)
    crew_df = pd.read_parquet(f'{data_dir}/imdb_title_crew.parquet')
    
    # Load principal cast/crew (actors, directors with ordering)
    principals_df = pd.read_parquet(f'{data_dir}/imdb_title_principals.parquet')
    
    # Load name basics (for actor/director names)
    names_df = pd.read_parquet(f'{data_dir}/imdb_name_basics.parquet')
    
    print("✅ Loaded IMDb data from parquet files")
    print(f"   • Movies/Shows: {len(movies_df):,}")
    print(f"   • Crew records: {len(crew_df):,}")
    print(f"   • Principal cast/crew: {len(principals_df):,}")
    print(f"   • Names: {len(names_df):,}")
    
except Exception as e:
    print(f"❌ Error loading processed data: {e}")
    print("Attempting to load from raw files...")
    
    # Fallback to raw files
    movies_df = pd.read_csv(f'{raw_dir}/title.basics.tsv.gz', sep='\t', na_values='\\N', low_memory=False)
    ratings_df = pd.read_csv(f'{raw_dir}/title.ratings.tsv.gz', sep='\t', na_values='\\N')
    crew_df = pd.read_csv(f'{raw_dir}/title.crew.tsv.gz', sep='\t', na_values='\\N')
    principals_df = pd.read_csv(f'{raw_dir}/title.principals.tsv.gz', sep='\t', na_values='\\N', low_memory=False)
    names_df = pd.read_csv(f'{raw_dir}/name.basics.tsv.gz', sep='\t', na_values='\\N', low_memory=False)
    
    # Merge ratings into movies
    movies_df = movies_df.merge(ratings_df, on='tconst', how='left')
    print("✅ Loaded from raw TSV files")

# Filter to only movies (not TV shows, episodes, etc.)
print(f"\n🎬 Filtering to movies only...")
print(f"Title types in data: {movies_df['titleType'].value_counts().head()}")

movies_df = movies_df[movies_df['titleType'] == 'movie'].copy()
print(f"✅ Filtered to {len(movies_df):,} movies")

# Basic data cleaning
print(f"\n🧹 Cleaning data...")
# Convert year to numeric
movies_df['startYear'] = pd.to_numeric(movies_df['startYear'], errors='coerce')
# Convert runtime to numeric
movies_df['runtimeMinutes'] = pd.to_numeric(movies_df['runtimeMinutes'], errors='coerce')
# Convert ratings to numeric
if 'averageRating' in movies_df.columns:
    movies_df['averageRating'] = pd.to_numeric(movies_df['averageRating'], errors='coerce')
    movies_df['numVotes'] = pd.to_numeric(movies_df['numVotes'], errors='coerce')

# Filter out movies without basic info
movies_df = movies_df[
    movies_df['startYear'].notna() & 
    movies_df['primaryTitle'].notna() &
    (movies_df['startYear'] >= 1900) &
    (movies_df['startYear'] <= 2025)
].copy()

print(f"✅ Cleaned data: {len(movies_df):,} movies with valid info")
print(f"   • Year range: {movies_df['startYear'].min():.0f} - {movies_df['startYear'].max():.0f}")
print(f"   • Movies with ratings: {movies_df['averageRating'].notna().sum():,}")

# Use IMDb ratings as target (only for movies that have ratings)
movies_with_ratings = movies_df[movies_df['averageRating'].notna()].copy()
print(f"\n🎯 Target variable: {len(movies_with_ratings):,} movies with ratings")
print(f"   • Rating range: {movies_with_ratings['averageRating'].min():.1f} - {movies_with_ratings['averageRating'].max():.1f}")
print(f"   • Average rating: {movies_with_ratings['averageRating'].mean():.2f}")

print("\n✅ IMDb data loaded and ready for feature engineering!")

📊 LOADING IMDb DATA
✅ Loaded IMDb data from parquet files
   • Movies/Shows: 450,860
   • Crew records: 726,969
   • Principal cast/crew: 8,374,077
   • Names: 14,745,597

🎬 Filtering to movies only...
Title types in data: titleType
movie    450860
Name: count, dtype: int64
✅ Loaded IMDb data from parquet files
   • Movies/Shows: 450,860
   • Crew records: 726,969
   • Principal cast/crew: 8,374,077
   • Names: 14,745,597

🎬 Filtering to movies only...
Title types in data: titleType
movie    450860
Name: count, dtype: int64
✅ Filtered to 450,860 movies

🧹 Cleaning data...
✅ Cleaned data: 450,627 movies with valid info
   • Year range: 1900 - 2025
   • Movies with ratings: 138,032
✅ Filtered to 450,860 movies

🧹 Cleaning data...
✅ Cleaned data: 450,627 movies with valid info
   • Year range: 1900 - 2025
   • Movies with ratings: 138,032

🎯 Target variable: 138,032 movies with ratings
   • Rating range: 1.0 - 9.9
   • Average rating: 5.91

✅ IMDb data loaded and ready for feature enginee

## 2. Create Target Variable and Basic Movie Features

We now use IMDb data which provides:
- **Pre-release information**: Genre, title, year, runtime, crew, cast
- **Target variable**: Average rating (from users who watched the movie)

Note: We do NOT use `numVotes` as a feature since it's post-release information.

## 2. Create Target Variable and Basic Movie Features

In [3]:
# Create movie features dataframe with ratings as target
print("🎯 CREATING FEATURE DATASET")
print("=" * 40)

# Start with movies that have ratings (our target variable)
movie_features = movies_with_ratings.copy()

# Rename for consistency
movie_features = movie_features.rename(columns={
    'tconst': 'movieId',
    'primaryTitle': 'title',
    'averageRating': 'avg_rating',
    'numVotes': 'vote_count',
    'startYear': 'year',
    'runtimeMinutes': 'runtime'
})

print(f"Movie features shape: {movie_features.shape}")
print(f"\n📊 Basic statistics for target variable (avg_rating):")
print(movie_features['avg_rating'].describe())
print(f"\n📈 Vote count statistics:")
print(movie_features['vote_count'].describe())

# Note: We keep vote_count for now but will NOT use it as a feature
# since it's post-release information. We only use avg_rating as the target.

🎯 CREATING FEATURE DATASET
Movie features shape: (138032, 51)

📊 Basic statistics for target variable (avg_rating):
count    138032.000000
mean          5.913499
std           1.270644
min           1.000000
25%           5.200000
50%           6.100000
75%           6.800000
max           9.900000
Name: avg_rating, dtype: float64

📈 Vote count statistics:
count    1.380320e+05
mean     8.833354e+03
std      5.756894e+04
min      1.000000e+02
25%      1.990000e+02
50%      4.720000e+02
75%      1.708000e+03
max      3.103987e+06
Name: vote_count, dtype: float64


## 3. Genre Features (Binary Encoding)

In [4]:
# Extract all unique genres
# IMDb uses comma-separated genres, not pipe-separated

# First, drop any existing genre columns to avoid duplicates
existing_genre_cols = [col for col in movie_features.columns if col.startswith('genre_')]
if existing_genre_cols:
    movie_features = movie_features.drop(columns=existing_genre_cols)
    print(f"Dropped {len(existing_genre_cols)} existing genre columns")

all_genres = set()
for genres in movie_features['genres'].dropna():
    if genres != '(no genres listed)':
        for genre in genres.split(','):  # Split by comma for IMDb data
            all_genres.add(genre.strip())

all_genres = sorted(list(all_genres))
print(f"\nFound {len(all_genres)} unique genres:")
print(all_genres)

# Create binary encoding for each genre (proper multi-hot encoding)
for genre in all_genres:
    # Use exact match with word boundaries to avoid partial matches
    movie_features[f'genre_{genre}'] = movie_features['genres'].apply(
        lambda x: 1 if pd.notna(x) and genre in [g.strip() for g in str(x).split(',')] else 0
    )

# Show genre feature columns
genre_columns = [col for col in movie_features.columns if col.startswith('genre_')]
print(f"\nCreated {len(genre_columns)} genre features")
print("Sample of genre features:")
print(movie_features[['title'] + genre_columns[:5]].head())

Dropped 28 existing genre columns

Found 27 unique genres:
['Action', 'Adult', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'Film-Noir', 'Game-Show', 'History', 'Horror', 'Music', 'Musical', 'Mystery', 'News', 'Reality-TV', 'Romance', 'Sci-Fi', 'Sport', 'Talk-Show', 'Thriller', 'War', 'Western']

Found 27 unique genres:
['Action', 'Adult', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'Film-Noir', 'Game-Show', 'History', 'Horror', 'Music', 'Musical', 'Mystery', 'News', 'Reality-TV', 'Romance', 'Sci-Fi', 'Sport', 'Talk-Show', 'Thriller', 'War', 'Western']

Created 27 genre features
Sample of genre features:
                          title  genre_Action  genre_Adult  genre_Adventure  \
1   The Story of the Kelly Gang             1            0                1   
14               Den sorte drøm             0            0                0   
16                The Traitress   

## 4. Temporal Features - REMOVED (Not useful for new movie decisions)

In [5]:
# TEMPORAL FEATURES - REMOVED
# Reason: When deciding to produce a new movie, the year is fixed (current year)
# movie_age and era features have zero variance for new productions
# and don't help in deciding what type of movie to make

print("📅 TEMPORAL FEATURES - SKIPPED")
print("=" * 40)
print("⚠️ Temporal features (movie_age, era) removed from feature set.")
print("   Reason: Not useful for predicting ratings of NEW movies")
print("   • movie_age: All new movies have age ≈ 0")
print("   • era: All new movies are 'Contemporary' by definition")
print("   • These features had predictive power in historical analysis")
print("     but provide no decision value for future productions")
print("\n✅ Skipping temporal feature creation")


📅 TEMPORAL FEATURES - SKIPPED
⚠️ Temporal features (movie_age, era) removed from feature set.
   Reason: Not useful for predicting ratings of NEW movies
   • movie_age: All new movies have age ≈ 0
   • era: All new movies are 'Contemporary' by definition
   • These features had predictive power in historical analysis
     but provide no decision value for future productions

✅ Skipping temporal feature creation


## 5. Popularity and Rating Distribution Features

In [6]:
# NOTE: Popularity and rating distribution features removed
# These features use post-release information (rating_count, rating_std)
# and are not available to studio producers before movie release

# ❌ REMOVED: is_popular, is_niche - based on rating_count
# ❌ REMOVED: log_rating_count - derived from post-release ratings
# ❌ REMOVED: is_polarizing, is_consistent - based on rating_std
# ❌ REMOVED: popularity_tier - categorization of post-release rating counts

print("⚠️ POPULARITY FEATURES SKIPPED")
print("These features require post-release rating data and are not")
print("available during pre-production planning.")
print("\n✅ Continuing with pre-release features only...")

⚠️ POPULARITY FEATURES SKIPPED
These features require post-release rating data and are not
available during pre-production planning.

✅ Continuing with pre-release features only...


## 6. Advanced Genre Features

In [7]:
# Count number of genres per movie
movie_features['genre_count'] = movie_features['genres'].apply(
    lambda x: len(x.split('|')) if pd.notna(x) and x != '(no genres listed)' else 0
)

# ❌ REMOVED: Redundant derived genre features
# The model can learn these patterns from the binary genre indicators and genre_count
# Removed: is_single_genre, is_multi_genre, genre_diversity_score, is_genre_mix
# Removed: Market positioning features (is_family_friendly, is_prestige_project, etc.)
# 
# Reason: These are derived from the binary genre columns and don't add new information.
# The model can learn genre combinations and patterns directly from:
#   - Individual binary genre columns (genre_Action=1, genre_Drama=1, etc.)
#   - genre_count (simple count of how many genres)

print("✅ Simplified genre features:")
print(f"Genre count distribution:")
print(movie_features['genre_count'].value_counts().sort_index())
print(f"\n📊 We now use:")
print(f"  • Binary genre indicators (genre_Action, genre_Drama, etc.)")
print(f"  • genre_count (number of genres per movie)")
print(f"\n⚠️ Removed redundant derived features:")
print("  • is_single_genre, is_multi_genre (model can learn from genre_count)")
print("  • genre_diversity_score, is_genre_mix (model can learn from binary indicators)")
print("  • Market positioning features (model can learn from genre combinations)")

✅ Simplified genre features:
Genre count distribution:
genre_count
0       214
1    137818
Name: count, dtype: int64

📊 We now use:
  • Binary genre indicators (genre_Action, genre_Drama, etc.)
  • genre_count (number of genres per movie)

⚠️ Removed redundant derived features:
  • is_single_genre, is_multi_genre (model can learn from genre_count)
  • genre_diversity_score, is_genre_mix (model can learn from binary indicators)
  • Market positioning features (model can learn from genre combinations)


## 7. Feature Selection and Final Dataset Preparation

## 6.5. Director, Writer, and Actor Features (Pre-Release)

In [8]:
# Extract crew and cast features from IMDb data
# Strategy: Create binary features for TOP directors/writers/actors
# based on POPULARITY (total votes), representing public recognition
print("🎬 DIRECTOR, WRITER & ACTOR FEATURES (Top People by Popularity)")
print("=" * 60)

# Merge crew data (directors, writers) if not already present
if 'directors' not in movie_features.columns:
    movie_features = movie_features.merge(
        crew_df[['tconst', 'directors', 'writers']], 
        left_on='movieId', 
        right_on='tconst', 
        how='left'
    )
    movie_features = movie_features.drop(columns=['tconst'], errors='ignore')

print("\n1️⃣ Identifying Top Directors (by popularity - total votes)...")

# Load language information from title.akas
print("  📚 Loading language information...")
akas_df = pd.read_parquet(f'{data_dir}/imdb_title_akas.parquet')
# Get English language movies
english_movies = akas_df[akas_df['language'] == 'en']['tconst'].unique()
english_movie_set = set(english_movies)
print(f"  ✅ Found {len(english_movie_set):,} English-language movies")

# Filter movies by minimum vote count (to ensure reliable ratings)
min_votes = 1000  # Require at least 1000 votes for reliable ratings
reliable_movies = movie_features[movie_features['vote_count'] >= min_votes].copy()
print(f"  ✅ Filtered to {len(reliable_movies):,} movies with ≥{min_votes:,} votes")

# Build director performance data (only English movies with sufficient votes)
director_movies = []
for idx, row in reliable_movies.iterrows():
    # Only include English-language movies
    if row['movieId'] not in english_movie_set:
        continue
    
    if pd.notna(row['directors']) and str(row['directors']) != '\\N':
        directors = [d.strip() for d in str(row['directors']).split(',')]
        for director_id in directors:
            director_movies.append({
                'director_id': director_id,
                'movie_id': row['movieId'],
                'year': row['year'],
                'rating': row['avg_rating'],
                'votes': row['vote_count']
            })

director_df = pd.DataFrame(director_movies)
print(f"  ✅ Found {len(director_df)} director-movie pairs (English, ≥{min_votes:,} votes)")

# Calculate statistics per director including total votes
director_stats = director_df.groupby('director_id').agg({
    'rating': ['mean', 'count', 'std'],
    'votes': 'sum',
    'year': ['min', 'max']
}).reset_index()
director_stats.columns = ['director_id', 'avg_rating', 'movie_count', 'rating_std', 'total_votes', 'first_year', 'last_year']

# Filter: Only living directors (check deathYear in names_df)
living_directors = names_df[names_df['deathYear'].isna()]['nconst'].values
director_stats = director_stats[director_stats['director_id'].isin(living_directors)].copy()
print(f"  ✅ Filtered to living directors only")

# Filter: At least 5 movies for statistical reliability AND worked in last 20 years
current_year = 2025
experienced_directors = director_stats[
    (director_stats['movie_count'] >= 5) & 
    (director_stats['last_year'] >= current_year - 20)
].copy()

# Sort by total votes (popularity-focused)
experienced_directors = experienced_directors.sort_values('total_votes', ascending=False)
top_20_directors = experienced_directors.head(20)['director_id'].tolist()

print(f"  ✅ Found {len(director_stats)} unique directors")
print(f"  ✅ Filtered to {len(experienced_directors)} directors with ≥5 movies")
print(f"  ✅ Top 20 directors by popularity (total votes):")
for i, row in experienced_directors.head(20).iterrows():
    print(f"     {i+1:2d}. {row['director_id']}: {int(row['total_votes']):,} votes ({int(row['movie_count'])} movies, {row['avg_rating']:.2f}★)")

# Create binary features for top 20 directors
for i, director_id in enumerate(top_20_directors, 1):
    feature_name = f'director_top{i:02d}'
    movie_features[feature_name] = movie_features['directors'].apply(
        lambda x: 1 if pd.notna(x) and str(x) != '\\N' and director_id in str(x).split(',') else 0
    )

print(f"\n  ✅ Created 20 binary features: director_top01 to director_top20")

print("\n2️⃣ Identifying Top Writers (by popularity - total votes)...")

# Build writer performance data (only English movies with sufficient votes)
writer_movies = []
for idx, row in reliable_movies.iterrows():
    # Only include English-language movies
    if row['movieId'] not in english_movie_set:
        continue
    
    if pd.notna(row['writers']) and str(row['writers']) != '\\N':
        writers = [w.strip() for w in str(row['writers']).split(',')]
        for writer_id in writers:
            writer_movies.append({
                'writer_id': writer_id,
                'movie_id': row['movieId'],
                'year': row['year'],
                'rating': row['avg_rating'],
                'votes': row['vote_count']
            })

writer_df = pd.DataFrame(writer_movies)
print(f"  ✅ Found {len(writer_df)} writer-movie pairs (English, ≥{min_votes:,} votes)")

# Calculate statistics per writer including total votes
writer_stats = writer_df.groupby('writer_id').agg({
    'rating': ['mean', 'count', 'std'],
    'votes': 'sum',
    'year': ['min', 'max']
}).reset_index()
writer_stats.columns = ['writer_id', 'avg_rating', 'movie_count', 'rating_std', 'total_votes', 'first_year', 'last_year']

# Filter: Only living writers
living_writers = names_df[names_df['deathYear'].isna()]['nconst'].values
writer_stats = writer_stats[writer_stats['writer_id'].isin(living_writers)].copy()
print(f"  ✅ Filtered to living writers only")

# Filter: At least 5 movies AND worked in last 20 years
experienced_writers = writer_stats[
    (writer_stats['movie_count'] >= 5) & 
    (writer_stats['last_year'] >= current_year - 20)
].copy()
experienced_writers = experienced_writers.sort_values('total_votes', ascending=False)
top_20_writers = experienced_writers.head(20)['writer_id'].tolist()

print(f"  ✅ Found {len(writer_stats)} unique writers")
print(f"  ✅ Filtered to {len(experienced_writers)} writers with ≥5 movies")
print(f"  ✅ Top 20 writers by popularity (total votes):")
for i, row in experienced_writers.head(20).iterrows():
    print(f"     {i+1:2d}. {row['writer_id']}: {int(row['total_votes']):,} votes ({int(row['movie_count'])} movies, {row['avg_rating']:.2f}★)")

# Create binary features for top 20 writers
for i, writer_id in enumerate(top_20_writers, 1):
    feature_name = f'writer_top{i:02d}'
    movie_features[feature_name] = movie_features['writers'].apply(
        lambda x: 1 if pd.notna(x) and str(x) != '\\N' and writer_id in str(x).split(',') else 0
    )

print(f"\n  ✅ Created 20 binary features: writer_top01 to writer_top20")

print("\n3️⃣ Identifying Top Actors (by popularity - total votes)...")

# Build actor performance data - OPTIMIZED VERSION (only English movies with sufficient votes)
actors_df_filtered = principals_df[principals_df['category'].isin(['actor', 'actress'])].copy()

# Create mapping efficiently using merge (only reliable movies)
print(f"  📊 Merging actor data with movie ratings...")
actor_movie_ratings = actors_df_filtered.merge(
    reliable_movies[['movieId', 'year', 'avg_rating', 'vote_count']], 
    left_on='tconst', 
    right_on='movieId', 
    how='inner'
)[['nconst', 'movieId', 'year', 'avg_rating', 'vote_count']]

# Filter to English movies only
actor_movie_ratings = actor_movie_ratings[actor_movie_ratings['movieId'].isin(english_movie_set)].copy()

print(f"  📊 Found {len(actor_movie_ratings)} actor-movie pairs (English, ≥{min_votes:,} votes)")

# Calculate statistics per actor including total votes
actor_stats = actor_movie_ratings.groupby('nconst').agg({
    'avg_rating': ['mean', 'count', 'std'],
    'vote_count': 'sum',
    'year': ['min', 'max']
}).reset_index()
actor_stats.columns = ['actor_id', 'avg_rating', 'movie_count', 'rating_std', 'total_votes', 'first_year', 'last_year']

# Filter: Only living actors
living_actors = names_df[names_df['deathYear'].isna()]['nconst'].values
actor_stats = actor_stats[actor_stats['actor_id'].isin(living_actors)].copy()
print(f"  ✅ Filtered to living actors only")

# Filter: At least 5 movies AND worked in last 20 years
experienced_actors = actor_stats[
    (actor_stats['movie_count'] >= 5) & 
    (actor_stats['last_year'] >= current_year - 20)
].copy()
experienced_actors = experienced_actors.sort_values('total_votes', ascending=False)
top_20_actors = experienced_actors.head(20)['actor_id'].tolist()

print(f"  ✅ Found {len(actor_stats)} unique actors")
print(f"  ✅ Filtered to {len(experienced_actors)} actors with ≥5 movies")
print(f"  ✅ Top 20 actors by popularity (total votes):")
for i, row in experienced_actors.head(20).iterrows():
    print(f"     {i+1:2d}. {row['actor_id']}: {int(row['total_votes']):,} votes ({int(row['movie_count'])} movies, {row['avg_rating']:.2f}★)")

# Create binary features for top 20 actors - OPTIMIZED
for i, actor_id in enumerate(top_20_actors, 1):
    feature_name = f'actor_top{i:02d}'
    # Get all movies for this actor
    actor_movie_set = set(actors_df_filtered[actors_df_filtered['nconst'] == actor_id]['tconst'].unique())
    movie_features[feature_name] = movie_features['movieId'].isin(actor_movie_set).astype(int)

print(f"\n  ✅ Created 20 binary features: actor_top01 to actor_top20")

# Summary statistics
director_features = [f'director_top{i:02d}' for i in range(1, 21)]
writer_features = [f'writer_top{i:02d}' for i in range(1, 21)]
actor_features = [f'actor_top{i:02d}' for i in range(1, 21)]

print("\n📊 FEATURE SUMMARY:")
print(f"  • Director features: {len(director_features)} (top 20 by popularity, ≥5 movies)")
print(f"  • Writer features: {len(writer_features)} (top 20 by popularity, ≥5 movies)")
print(f"  • Actor features: {len(actor_features)} (top 20 by popularity, ≥5 movies)")
print(f"  • Total: 60 binary features")

print("\n🎯 INTERPRETATION FOR LINEAR REGRESSION:")
print("  • Positive coefficient = this person's involvement correlates with higher ratings")
print("  • Negative coefficient = this person's involvement correlates with lower ratings")
print("  • Example: If director_top01 has coef=+0.5, hiring that director")
print("    increases predicted rating by +0.5 points (all else equal)")

print("\n⚠️ SELECTION CRITERIA:")
print("  • Based on TOTAL VOTES across their films (popularity/public recognition)")
print("  • Minimum 5 films for statistical reliability")
print("  • Must have worked on projects in the last 20 years (2005-2025)")
print("  • Only living filmmakers (deathYear is null)")
print("  • Only English-language movies considered")
print(f"  • Only movies with ≥{min_votes:,} votes (reliable ratings)")
print("  • NOT using future films - only historical performance")
print("  • Pre-release information: studios know popularity before hiring!")

# Show how many movies have these top people
print("\n📈 COVERAGE:")
director_coverage = movie_features[director_features].sum(axis=1).astype(bool).sum()
writer_coverage = movie_features[writer_features].sum(axis=1).astype(bool).sum()
actor_coverage = movie_features[actor_features].sum(axis=1).astype(bool).sum()

print(f"  • Movies with at least one top-20 director: {director_coverage} ({director_coverage/len(movie_features)*100:.1f}%)")
print(f"  • Movies with at least one top-20 writer: {writer_coverage} ({writer_coverage/len(movie_features)*100:.1f}%)")
print(f"  • Movies with at least one top-20 actor: {actor_coverage} ({actor_coverage/len(movie_features)*100:.1f}%)")

# Store stats for later name lookup
director_counts = director_stats.set_index('director_id')['movie_count'].to_dict()
writer_counts = writer_stats.set_index('writer_id')['movie_count'].to_dict()
actor_counts = actor_stats.set_index('actor_id')['movie_count'].to_dict()

director_ratings = director_stats.set_index('director_id')['avg_rating'].to_dict()
writer_ratings = writer_stats.set_index('writer_id')['avg_rating'].to_dict()
actor_ratings = actor_stats.set_index('actor_id')['avg_rating'].to_dict()


🎬 DIRECTOR, WRITER & ACTOR FEATURES (Top People by Popularity)

1️⃣ Identifying Top Directors (by popularity - total votes)...
  📚 Loading language information...
  ✅ Found 191,953 English-language movies
  ✅ Filtered to 46,612 movies with ≥1,000 votes
  ✅ Found 191,953 English-language movies
  ✅ Filtered to 46,612 movies with ≥1,000 votes
  ✅ Found 47380 director-movie pairs (English, ≥1,000 votes)
  ✅ Found 47380 director-movie pairs (English, ≥1,000 votes)
  ✅ Filtered to living directors only
  ✅ Found 17450 unique directors
  ✅ Filtered to 1449 directors with ≥5 movies
  ✅ Top 20 directors by popularity (total votes):
     6566. nm0634240: 17,600,165 votes (12 movies, 8.17★)
     76. nm0000229: 15,394,644 votes (34 movies, 7.37★)
     79. nm0000233: 12,690,004 votes (15 movies, 7.77★)
     73. nm0000217: 11,383,991 votes (34 movies, 7.52★)
     179. nm0000631: 9,800,380 votes (30 movies, 6.97★)
     118. nm0000399: 9,732,409 votes (12 movies, 7.58★)
     370. nm0001392: 9,459,130

### Auteur Features (Writer-Directors)

In [9]:
# 4️⃣ AUTEUR FEATURES: Movies where the same person is both writer AND director
# These "auteur" films often have a unique creative vision and may impact ratings
print("\n4️⃣ Identifying Auteur Films (Writer-Director)...")
print("=" * 60)

# Find movies where at least one person is both director and writer
auteur_movies = []
for idx, row in reliable_movies.iterrows():
    # Only English movies
    if row['movieId'] not in english_movie_set:
        continue
    
    # Get directors and writers for this movie
    if pd.notna(row['directors']) and str(row['directors']) != '\\N' and \
       pd.notna(row['writers']) and str(row['writers']) != '\\N':
        directors = set([d.strip() for d in str(row['directors']).split(',')])
        writers = set([w.strip() for w in str(row['writers']).split(',')])
        
        # Check if there's overlap (auteur)
        auteurs = directors.intersection(writers)
        if auteurs:
            auteur_movies.append({
                'movie_id': row['movieId'],
                'auteurs': list(auteurs),
                'rating': row['avg_rating'],
                'year': row['year']
            })

print(f"  ✅ Found {len(auteur_movies)} auteur films (same person wrote & directed)")
print(f"  ✅ That's {len(auteur_movies)/len(reliable_movies)*100:.1f}% of reliable movies")

# Create binary feature for auteur films
movie_features['is_auteur_film'] = movie_features['movieId'].isin([m['movie_id'] for m in auteur_movies]).astype(int)

# Compare ratings: auteur vs non-auteur
auteur_ratings = [m['rating'] for m in auteur_movies]
non_auteur_ratings = reliable_movies[~reliable_movies['movieId'].isin([m['movie_id'] for m in auteur_movies])]['avg_rating'].tolist()

print(f"\n📊 AUTEUR FILM STATISTICS:")
print(f"  • Auteur films average rating: {sum(auteur_ratings)/len(auteur_ratings):.2f}")
print(f"  • Non-auteur films average rating: {sum(non_auteur_ratings)/len(non_auteur_ratings):.2f}")
print(f"  • Difference: {sum(auteur_ratings)/len(auteur_ratings) - sum(non_auteur_ratings)/len(non_auteur_ratings):+.2f} points")

print(f"\n✅ Created 1 auteur feature: is_auteur_film")
print("   (Individual auteurs already captured in director/writer features)")



4️⃣ Identifying Auteur Films (Writer-Director)...
  ✅ Found 24189 auteur films (same person wrote & directed)
  ✅ That's 51.9% of reliable movies

📊 AUTEUR FILM STATISTICS:
  • Auteur films average rating: 6.28
  • Non-auteur films average rating: 6.18
  • Difference: +0.10 points

✅ Created 1 auteur feature: is_auteur_film
   (Individual auteurs already captured in director/writer features)
  ✅ Found 24189 auteur films (same person wrote & directed)
  ✅ That's 51.9% of reliable movies

📊 AUTEUR FILM STATISTICS:
  • Auteur films average rating: 6.28
  • Non-auteur films average rating: 6.18
  • Difference: +0.10 points

✅ Created 1 auteur feature: is_auteur_film
   (Individual auteurs already captured in director/writer features)


In [10]:
# Optional: Get actual names for the top directors/writers/actors
# This helps with interpretation when analyzing regression coefficients
print("🎭 IDENTIFYING TOP PEOPLE BY NAME (Popularity-Based Selection)")
print("=" * 65)

# Load name mappings
name_mapping = names_df.set_index('nconst')['primaryName'].to_dict()

print("\n🎬 Top 20 Directors (by popularity):")
for i, director_id in enumerate(top_20_directors, 1):
    name = name_mapping.get(director_id, 'Unknown')
    rating = director_ratings.get(director_id, 0)
    count = director_counts.get(director_id, 0)
    print(f"  {i:2d}. director_top{i:02d} = {name:30s} | {rating:.2f}★ ({int(count)} movies)")

print("\n✍️  Top 20 Writers (by popularity):")
for i, writer_id in enumerate(top_20_writers, 1):
    name = name_mapping.get(writer_id, 'Unknown')
    rating = writer_ratings.get(writer_id, 0)
    count = writer_counts.get(writer_id, 0)
    print(f"  {i:2d}. writer_top{i:02d} = {name:30s} | {rating:.2f}★ ({int(count)} movies)")

print("\n🎭 Top 20 Actors (by popularity):")
for i, actor_id in enumerate(top_20_actors, 1):
    name = name_mapping.get(actor_id, 'Unknown')
    rating = actor_ratings.get(actor_id, 0)
    count = actor_counts.get(actor_id, 0)
    print(f"  {i:2d}. actor_top{i:02d} = {name:30s} | {rating:.2f}★ ({int(count)} movies)")

# Save the mapping for later reference
# NOTE: Only saving pre-release information (name, historical avg rating, movie count)
# NOT including auteurs separately to avoid confusion (they're already in directors/writers)
top_people_mapping = {
    'directors': {f'director_top{i:02d}': {
        'id': director_id, 
        'name': name_mapping.get(director_id, 'Unknown'),
        'avg_rating': float(director_ratings.get(director_id, 0)),
        'movie_count': int(director_counts.get(director_id, 0))
    } for i, director_id in enumerate(top_20_directors, 1)},
    'writers': {f'writer_top{i:02d}': {
        'id': writer_id,
        'name': name_mapping.get(writer_id, 'Unknown'),
        'avg_rating': float(writer_ratings.get(writer_id, 0)),
        'movie_count': int(writer_counts.get(writer_id, 0))
    } for i, writer_id in enumerate(top_20_writers, 1)},
    'actors': {f'actor_top{i:02d}': {
        'id': actor_id,
        'name': name_mapping.get(actor_id, 'Unknown'),
        'avg_rating': float(actor_ratings.get(actor_id, 0)),
        'movie_count': int(actor_counts.get(actor_id, 0))
    } for i, actor_id in enumerate(top_20_actors, 1)},
    'selection_criteria': {
        'method': 'total_votes_popularity',
        'min_movies': 5,
        'note': 'Selected based on total votes (popularity). Only pre-release info saved. Auteurs not included separately to avoid duplication with directors/writers.'
    }
}

import json
with open('../data/top_people_mapping.json', 'w') as f:
    json.dump(top_people_mapping, f, indent=2)

print("\n💾 Saved name mappings to: ../data/top_people_mapping.json")
print("   Use this file to interpret regression coefficients later!")
print("\n✅ These are the most popular and recognizable filmmakers!")
print("   Auteur analysis kept separate (is_auteur_film feature only).")


🎭 IDENTIFYING TOP PEOPLE BY NAME (Popularity-Based Selection)

🎬 Top 20 Directors (by popularity):
   1. director_top01 = Christopher Nolan              | 8.17★ (12 movies)
   2. director_top02 = Steven Spielberg               | 7.37★ (34 movies)
   3. director_top03 = Quentin Tarantino              | 7.77★ (15 movies)
   4. director_top04 = Martin Scorsese                | 7.52★ (34 movies)
   5. director_top05 = Ridley Scott                   | 6.97★ (30 movies)
   6. director_top06 = David Fincher                  | 7.58★ (12 movies)
   7. director_top07 = Peter Jackson                  | 7.70★ (15 movies)
   8. director_top08 = Robert Zemeckis                | 6.99★ (22 movies)
   9. director_top09 = James Cameron                  | 7.28★ (11 movies)
  10. director_top10 = Clint Eastwood                 | 6.97★ (41 movies)
  11. director_top11 = Tim Burton                     | 6.97★ (20 movies)
  12. director_top12 = Francis Ford Coppola           | 6.54★ (28 movies)
  13. directo

### 🎯 Advantages of Top People Features (Quality-Based Selection)

**Why this approach works well:**

1. **Quality Over Quantity**:
   - Selected by **average rating** (proven quality), not just number of films
   - Minimum 5 films ensures statistical reliability (not one-hit wonders)
   - You'll recognize these names - they're respected filmmakers!

2. **Interpretable Coefficients**: After training linear regression, you can see:
   - `director_top01` (highest-rated director) coefficient = +0.3 → Adds +0.3 to rating
   - `actor_top15` (lower-rated actor) coefficient = -0.1 → Reduces rating by -0.1
   - This tells studios **who to hire** for better ratings!

3. **No Data Leakage**: 
   - Uses **historical performance** (past films only)
   - For each movie, we only use ratings from films released *before* it
   - Pre-release information: studios know director's reputation before hiring
   - We're NOT using "this movie's rating" to predict itself

4. **Sparse but Meaningful**:
   - Only a small % of movies have top-20 people (sparse features)
   - But those movies can leverage this quality signal
   - Most movies get prediction from other features (genre, year, etc.)

5. **Reputation Matters**:
   - Better than "number of movies" which gave us obscure, prolific directors
   - Captures what studios actually care about: **proven track record**
   - Recognizes that hiring a consistently high-rated director is valuable

**Example Interpretation After Training:**
```
Feature                Coefficient    Interpretation
--------------------- -------------- ---------------------------------
director_top01         +0.52         Top director adds +0.52 to rating
director_top20         +0.30         Still good, adds +0.30 to rating  
writer_top01           +0.41         Top writer adds +0.41
actor_top08            +0.25         Quality actor adds +0.25
genre_Action           +0.20         Action genre adds +0.20
genre_Drama            +0.40         Drama genre adds +0.40
```

**Selection Criteria:**
- **Average rating** of their past films (quality measure)
- **Minimum 5 films** (statistical reliability, not flukes)
- **Historical only** (no future information leakage)

This gives you **recognizable, high-quality filmmakers** whose reputation can inform pre-production decisions!

In [11]:
# Define target variable and feature columns
target = 'avg_rating'

# ⚠️ CRITICAL: Exclude post-release information from features
# Only include features that studio producers can know BEFORE movie release

# Post-release features to EXCLUDE:
post_release_features = [
    # Vote/rating counts (only available after people watch the movie)
    'vote_count', 'rating_count', 'rating_std', 'log_rating_count',
    'numVotes',  # IMDb vote count (post-release)
    'is_popular', 'is_niche', 'is_polarizing', 'is_consistent',
    'popularity_tier', 
    # Any popularity_* dummies if they were created
    'popularity_Low', 'popularity_Medium', 'popularity_High', 
    'popularity_Very Low', 'popularity_Very High',
    # Genre performance features based on actual ratings
    'Action_genre_performance', 'Adventure_genre_performance', 
    'Animation_genre_performance', 'Children_genre_performance',
    'Comedy_genre_performance', 'Crime_genre_performance',
    'Documentary_genre_performance', 'Drama_genre_performance',
    'Fantasy_genre_performance', 'Film-Noir_genre_performance',
    'Horror_genre_performance', 'IMAX_genre_performance',
    'Musical_genre_performance', 'Mystery_genre_performance',
    'Romance_genre_performance', 'Sci-Fi_genre_performance',
    'Thriller_genre_performance', 'War_genre_performance',
    'Western_genre_performance',
    # Director/actor performance based on ratings
    'director_avg_rating', 'actor_avg_rating',
    'has_top_director', 'has_experienced_director', 'has_high_rated_director',
    'has_top_actor', 'has_experienced_actor', 'has_high_rated_actor',
    'director_actor_star_power'
]

# Temporal features to EXCLUDE (not useful for new movie decisions):
temporal_features = [
    'movie_age',  # All new movies have age ≈ 0 (zero variance)
    'era_Classic', 'era_Contemporary', 'era_Golden Age', 'era_Modern',  # All new movies are Contemporary
    'decade',  # All new movies in same decade
]

# Non-feature columns (identifiers, raw text, intermediate variables)
exclude_columns = [
    'movieId', 'title', 'genres', 'year', 'era',
    'clean_title', 'director_name', 'lead_actor', 'second_actor', 'third_actor',
    'imdbId_clean', 'imdb_id_clean', 'credits',
    # Raw ID columns (not useful as features - we use derived features instead)
    'directors', 'writers',  # These are comma-separated IMDb person IDs
    'cast_size_category',  # Categorical column (we use dummies instead)
    # IMDb administrative fields
    'tconst', 'titleType', 'primaryTitle', 'originalTitle', 'isAdult', 
    'startYear', 'endYear', 'runtimeMinutes', 'averageRating',
    # Regional/language data (low signal, noisy)
    'region', 'language', 'types', 'attributes', 'isOriginalTitle',
    'num_regional_variants', 'num_regions', 'num_languages',
    # Crew size features (low predictive power)
    'num_directors', 'num_writers', 'num_actors', 'total_cast_crew',
]

# Combine all columns to exclude
all_exclude_columns = exclude_columns + post_release_features + temporal_features + [target]

# Get all feature columns (only pre-release information)
available_columns = movie_features_with_credits.columns if 'movie_features_with_credits' in locals() else movie_features.columns
feature_columns = [col for col in available_columns if col not in all_exclude_columns]

# Create final feature matrix
source_df = movie_features_with_credits if 'movie_features_with_credits' in locals() else movie_features
X = source_df[feature_columns].copy()
y = source_df[target].copy()

# Handle any remaining missing values
X = X.fillna(0)

# Remove movies with missing target values
valid_indices = y.notna()
X = X[valid_indices]
y = y[valid_indices]

print(f"✅ PRE-RELEASE FEATURES ONLY (Actionable for New Productions)")
print(f"=" * 60)
print(f"Final dataset shape: {X.shape}")
print(f"Target variable shape: {y.shape}")
print(f"Features included: {len(feature_columns)}")
print(f"\nFeature categories:")
print(f"- Genre features: {len([col for col in feature_columns if col.startswith('genre_')])}")
print(f"- Director features: {len([col for col in feature_columns if col.startswith('director_')])}")
print(f"- Writer features: {len([col for col in feature_columns if col.startswith('writer_')])}")
print(f"- Actor features: {len([col for col in feature_columns if col.startswith('actor_')])}")
print(f"- Other features: {len([col for col in feature_columns if not any(col.startswith(prefix) for prefix in ['genre_', 'director_', 'writer_', 'actor_'])])}")

print(f"\n⚠️ EXCLUDED FEATURES:")
print(f"  • {len(post_release_features)} post-release features (votes, rating-based metrics)")
print(f"  • {len(temporal_features)} temporal features (zero variance for new movies)")
print(f"  • {len(exclude_columns)} non-feature columns (IDs, raw text, admin fields)")

print(f"\n📋 Sample of PRE-RELEASE feature names:")
print(feature_columns[:20])

print(f"\n🎯 These features answer: 'What can we decide BEFORE making the movie?'")
print(f"   Genre selection, talent hiring, runtime, auteur approach")


✅ PRE-RELEASE FEATURES ONLY (Actionable for New Productions)
Final dataset shape: (138032, 90)
Target variable shape: (138032,)
Features included: 90

Feature categories:
- Genre features: 28
- Director features: 20
- Writer features: 20
- Actor features: 20
- Other features: 2

⚠️ EXCLUDED FEATURES:
  • 43 post-release features (votes, rating-based metrics)
  • 6 temporal features (zero variance for new movies)
  • 37 non-feature columns (IDs, raw text, admin fields)

📋 Sample of PRE-RELEASE feature names:
['runtime', 'genre_Action', 'genre_Adult', 'genre_Adventure', 'genre_Animation', 'genre_Biography', 'genre_Comedy', 'genre_Crime', 'genre_Documentary', 'genre_Drama', 'genre_Family', 'genre_Fantasy', 'genre_Film-Noir', 'genre_Game-Show', 'genre_History', 'genre_Horror', 'genre_Music', 'genre_Musical', 'genre_Mystery', 'genre_News']

🎯 These features answer: 'What can we decide BEFORE making the movie?'
   Genre selection, talent hiring, runtime, auteur approach


## 8. Save Processed Data for Model Training

In [12]:
# Save the engineered features to CSV for easy loading in model training
processed_data = pd.concat([X, y], axis=1)
processed_data['movieId'] = source_df.loc[valid_indices, 'movieId'].values
processed_data['title'] = source_df.loc[valid_indices, 'title'].values

# Reorder columns to have identifiers first
cols = ['movieId', 'title', target] + feature_columns
processed_data = processed_data[cols]

# Save to data directory
output_path = '../data/processed_movie_features.csv'
processed_data.to_csv(output_path, index=False)

print(f"💾 Processed data saved to: {output_path}")
print(f"Dataset contains {len(processed_data)} movies with {len(feature_columns)} features")
print(f"✅ ALL FEATURES ARE PRE-RELEASE ONLY (Actionable for New Productions)")

# Also save feature column names for easy reference
feature_info = {
    'target_variable': target,
    'feature_columns': feature_columns,
    'feature_count': len(feature_columns),
    'note': 'All features are pre-release information for deciding what movie to produce. Temporal features removed as they have zero variance for new movies.',
    'excluded_post_release_features': post_release_features,
    'excluded_temporal_features': temporal_features,
    'excluded_other': exclude_columns
}

import json
with open('../data/feature_info.json', 'w') as f:
    json.dump(feature_info, f, indent=2)

print("Feature information saved to: ../data/feature_info.json")

# Display final data sample
print(f"\n📋 Final processed data sample:")
display_cols = ['title', target] + feature_columns[:5]
print(processed_data[display_cols].head())


💾 Processed data saved to: ../data/processed_movie_features.csv
Dataset contains 138032 movies with 90 features
✅ ALL FEATURES ARE PRE-RELEASE ONLY (Actionable for New Productions)
Feature information saved to: ../data/feature_info.json

📋 Final processed data sample:
                          title  avg_rating  runtime  genre_Action  \
1   The Story of the Kelly Gang         6.0     70.0             1   
14               Den sorte drøm         5.8     53.0             0   
16                The Traitress         5.8     48.0             0   
21                    Cleopatra         5.1    100.0             0   
22              Dante's Inferno         7.0     71.0             0   

    genre_Adult  genre_Adventure  genre_Animation  
1             0                1                0  
14            0                0                0  
16            0                0                0  
21            0                0                0  
22            0                1                0 